# 可选实验：简单的 Neural Network

本实验将使用 NumPy 构建一个小型 neural network，其结构与之前用 TensorFlow 实现的 Coffee Roasting Network 相同。

<figure style="text-align:center;">
  <img src="./images/C2_W1_CoffeeRoasting.png" alt="咖啡烘焙示例" width="400">
</figure>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from lab_utils_common import dlc, sigmoid
from lab_coffee_utils import load_coffee_data, plt_roast, plt_prob, plt_layer, plt_network, plt_output_unit
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## 数据集

我们使用与上一个实验相同的数据集。


In [ ]:
X,Y = load_coffee_data();
print(X.shape, Y.shape)

下面绘制咖啡烘焙数据。两个 feature 分别是温度（摄氏度）和时长（分钟）。本实验参考的 [Coffee Roasting at Home](https://www.merchantsofgreencoffee.com/how-to-roast-green-coffee-in-your-oven/) 将烘焙时长设在 12–15 分钟、温度设在 175–260 摄氏度的范围内。温度升高时，合适的时长应相应缩短。


In [ ]:
plt_roast(X,Y)

### 对数据进行 Normalization

为与上一个实验保持一致，先对数据进行 normalization。详细说明请参阅上一个实验。


In [ ]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)  # learns mean, variance
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

## NumPy 模型：Forward Propagation

<figure style="text-align:center;">
  <img src="./images/C2_W1_RoastingNetwork.PNG" alt="两层咖啡烘焙 neural network" width="200">
</figure>

下面构建课程中的 Coffee Roasting Network，它包含两个使用 sigmoid activation 的 layer。


如课程所述，可以使用 NumPy 自行实现 dense layer（全连接层），再用它搭建多层 neural network。

<img src="images/C2_W1_dense3.png" alt="Dense layer 的计算过程" width="600" height="450">

第一个可选实验已经比较了 NumPy 与 TensorFlow 中的 neuron。Layer 只是包含多个 neuron：通过 `for` loop 遍历每个 unit `j`，将输入与该 unit 的 weight `W[:, j]` 做 dot product，再加上 `b[j]` 得到 `z`，最后计算 activation function `g(z)`。下面将这些步骤封装为 dense layer 函数。


先定义 activation function `g()`。这里使用 `lab_utils_common.py` 中已经提供的 `sigmoid()`。


In [ ]:
# Define the activation function
g = sigmoid

接着定义 `my_dense()`，计算一个 dense layer 的 activation。


In [ ]:
def my_dense(a_in, W, b):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example 
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units  
    Returns
      a_out (ndarray (j,))  : j units|
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):               
        w = W[:,j]                                    
        z = np.dot(w, a_in) + b[j]         
        a_out[j] = g(z)               
    return(a_out)

> **注意：** 也可以将 `g` 作为参数传入，例如 `my_dense(a_in, W, b, g)`。本实验只使用 sigmoid，因此将其定义在函数外也可以，且后续调用更简洁。两种方式都有效；本周的编程作业会使用将 `g` 作为参数传入的形式。


下面使用 `my_dense` 构建一个两层 neural network。


In [ ]:
def my_sequential(x, W1, b1, W2, b2):
    a1 = my_dense(x,  W1, b1)
    a2 = my_dense(a1, W2, b2)
    return(a2)

可以直接使用上一个 TensorFlow 实验训练得到的 weight 和 bias。


In [ ]:
W1_tmp = np.array( [[-8.93,  0.29, 12.9 ], [-0.1,  -7.32, 10.81]] )
b1_tmp = np.array( [-9.82, -9.28,  0.96] )
W2_tmp = np.array( [[-31.18], [-27.59], [-32.56]] )
b2_tmp = np.array( [15.41] )

### Predictions

<img align="left" src="./images/C2_W1_RoastingDecision.PNG" alt="根据烘焙成功概率进行分类" style="width:380px;padding:10px 20px;">

训练好模型后，就可以进行预测。这里的输出表示烘焙效果良好的概率。若要给出类别决策，需要应用 threshold；本例使用 0.5。


下面编写一个类似 TensorFlow `model.predict()` 的函数。它接收 matrix $X$，每行一个样本，共 $m$ 个样本，然后运行模型获得预测。


In [ ]:
def my_predict(X, W1, b1, W2, b2):
    m = X.shape[0]
    p = np.zeros((m,1))
    for i in range(m):
        p[i,0] = my_sequential(X[i], W1, b1, W2, b2)
    return(p)

下面使用两个样本测试该函数。


In [ ]:
X_tst = np.array([
    [200,13.9],  # postive example
    [200,17]])   # negative example
X_tstn = norm_l(X_tst)  # remember to normalize
predictions = my_predict(X_tstn, W1_tmp, b1_tmp, W2_tmp, b2_tmp)

使用 threshold，将概率转换为类别决策：


In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

也可以用更简洁的写法完成：


In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

## 整个网络的行为


这组图展示了整个网络的行为，与上一个 TensorFlow 实验的结果相同。

- 左图：蓝色深浅表示最后一层的原始输出，叠加的“X”和“O”是训练数据。
- 右图：应用 threshold 后的类别预测，“X”和“O”表示网络作出的决策。


In [ ]:
netf= lambda x : my_predict(norm_l(x),W1_tmp, b1_tmp, W2_tmp, b2_tmp)
plt_network(X,Y,netf)

## 恭喜完成实验！

你使用 NumPy 构建了一个小型 neural network，并看到了 neural network 中的 layer 是如何由这些简单、熟悉的计算组成的。
